# 재난문자 63,402건 × 기상청 날씨: 같은 더위에도 문자는 더 많이 왔을까

데이터팝콘 테크블로그 [글](https://datapopcorn.ai/blog/disaster-alerts-weather-analysis)의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글의 모든 숫자와 차트 7장이 다시 만들어집니다. API 키는 필요 없습니다. 데이터는 공개 저장소 `team-datapopcorn/datapopcorn-assets/disaster-alerts-weather/data`에서 읽습니다.

## Step 0. 준비: 폰트와 데이터 경로

In [ ]:
import os, json, urllib.request
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

BASE = 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/disaster-alerts-weather/'
HERE = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '.'
def path(p):
    local = os.path.join(HERE, p)
    return local if os.path.exists(local) else BASE + p

os.makedirs(os.path.join(HERE, 'fonts'), exist_ok=True)
for w in ['Regular', 'SemiBold', 'Bold']:
    f = os.path.join(HERE, 'fonts', f'Pretendard-{w}.otf')
    if not os.path.exists(f):
        urllib.request.urlretrieve(f'https://cdn.jsdelivr.net/gh/orioncactus/pretendard@v1.3.9/packages/pretendard/dist/public/static/Pretendard-{w}.otf', f)
    fm.fontManager.addfont(f)
plt.rcParams['font.family'] = 'Pretendard'
plt.rcParams['axes.unicode_minus'] = False
OUT = os.path.join(HERE, 'charts'); os.makedirs(OUT, exist_ok=True)
R = {}  # 글에 쓰는 숫자를 모두 여기에 모읍니다

## Step 1. 데이터 불러오기

In [ ]:
alerts = pd.read_csv(path('data/alerts.csv.gz'))
asos = pd.read_csv(path('data/asos_daily.csv.gz'))
stn = pd.read_csv(path('data/station_sido.csv'))
R['n_alerts'] = int(len(alerts)); R['n_alert_unique_sn'] = int(alerts.SN.nunique())
R['alert_range'] = [alerts.date.min(), alerts.date.max()]
R['n_station_days'] = int(len(asos)); R['n_stations'] = int(asos.stnId.nunique())
assert R['n_alerts'] == R['n_alert_unique_sn'], '일련번호(SN) 중복이 있습니다'
print(R)

## Step 2. 뉴스 숫자와 맞춰 보기 (행정안전부 발표값 교차검증)

In [ ]:
heat = alerts[alerts.DST_SE_NM == '폭염']
R['heat_2026_jan_jul'] = int(((heat.date >= '2026-01-01') & (heat.date <= '2026-07-31')).sum())
R['heat_2026_0804_0810'] = int(((heat.date >= '2026-08-04') & (heat.date <= '2026-08-10')).sum())
print('2026년 1~7월 폭염 문자:', R['heat_2026_jan_jul'], '(행안부 발표 2,563건)')
print('2026년 8월 4~10일 폭염 문자:', R['heat_2026_0804_0810'], '(행안부 발표 1,292건)')

## Step 3. 데이터가 비어 있는 기간 찾기

In [ ]:
idx = pd.date_range('2023-09-01', R['alert_range'][1]).strftime('%Y-%m-%d')
daily = alerts.groupby('date').size().reindex(idx, fill_value=0)
runs, start, prev = [], None, None
for d, v in daily.items():
    if v == 0 and start is None: start = d
    if v != 0 and start is not None: runs.append((start, prev)); start = None
    prev = d
runs = [r for r in runs if (pd.Timestamp(r[1]) - pd.Timestamp(r[0])).days >= 3]
R['zero_runs'] = runs
GAP = ('2025-09-26', '2025-10-26')
R['gap_days'] = int((pd.Timestamp(GAP[1]) - pd.Timestamp(GAP[0])).days + 1)
R['oct2025_rows'] = int(alerts.date.str.startswith('2025-10').sum())
print('4일 이상 연속 0건 구간:', runs)

## Step 4. 시도×일 패널 만들기 (섬 관측소 3곳 제외, 시도 안 관측소 중 최댓값)

In [ ]:
w = asos[~asos.stnId.isin(stn.loc[stn.island_excluded, 'stnId'])]
panel = (w.groupby(['sido', 'tm'])
          .agg(tmax=('maxTa', 'max'), feels=('feelsMax', 'max'), feels_mean=('feelsMax', 'mean'), tmin=('minTa', 'min'), rain=('sumRn', 'max'))
          .reset_index().rename(columns={'tm': 'date'}))
ex = alerts.assign(sido=alerts.sido_list.str.split('|')).explode('sido')
for typ, col in [('폭염', 'heat'), ('호우', 'rain_alert'), ('한파', 'cold')]:
    c = ex[ex.DST_SE_NM == typ].groupby(['sido', 'date']).size().rename(col)
    panel = panel.merge(c, left_on=['sido', 'date'], right_index=True, how='left')
panel[['heat', 'rain_alert', 'cold']] = panel[['heat', 'rain_alert', 'cold']].fillna(0)
panel['year'] = panel.date.str[:4].astype(int); panel['month'] = panel.date.str[5:7].astype(int)
panel = panel[~panel.date.between(*GAP)]            # 데이터 공백 기간 제외
R['panel_rows'] = int(len(panel)); R['n_sido'] = int(panel.sido.nunique())

## Step 5. 체감온도별 폭염 문자 확률 (2024~2026년 6~9월)

In [ ]:
S = panel[panel.month.isin([6, 7, 8, 9]) & panel.year.isin([2024, 2025, 2026])].dropna(subset=['feels']).copy()
S['hit'] = (S.heat > 0).astype(int)
S['fbin'] = np.clip(np.floor(S.feels), 26, 37).astype(int)
dose = S.groupby('fbin').agg(p=('hit', 'mean'), n=('hit', 'size')).reset_index()
hot = S[S.feels >= 33]
R['p_alert_hot'] = round(float(hot.hit.mean()), 3); R['n_hot_sido_days'] = int(len(hot))
R['p_alert_lt29'] = round(float(S[S.feels < 29].hit.mean()), 3)
R['share_heat_alertdays_feels_lt31'] = round(float((S[S.hit == 1].feels < 31).mean()), 3)
print(dose)

## Step 6. 같은 더위, 다른 문자 수: 연도 비교

In [ ]:
by_year = hot.groupby('year').agg(hot_days=('hit', 'size'), p_any=('hit', 'mean'), per_hot_day=('heat', 'mean'),
                                  feels_mean=('feels', 'mean')).round(3)
by_year['per_alert_day'] = hot[hot.hit == 1].groupby('year').heat.mean().round(2)
R['by_year'] = json.loads(by_year.to_json(orient='index'))
S['fb36'] = np.clip(S.fbin, 26, 36)
curve = S.groupby(['year', 'fb36']).heat.mean().unstack(0)
R['curve_n'] = {int(k[0]) * 100 + int(k[1]): int(v) for k, v in S.groupby(['year', 'fb36']).size().items()}
print(by_year)

## Step 7. 다른 조건을 같게 놓고 비교 (포아송 회귀)

In [ ]:
import statsmodels.formula.api as smf
S = S.sort_values(['sido', 'date'])
S['hot'] = (S.feels >= 33).astype(int)
S['run'] = S.groupby('sido').hot.transform(lambda x: x.groupby((x != x.shift()).cumsum()).cumsum()) * S.hot
S['run10'] = np.minimum(S.run, 10)
groups = S.sido.astype('category').cat.codes
m = smf.glm('heat ~ C(year) + C(fbin) + C(sido) + C(month) + run10', data=S,
            family=__import__('statsmodels.api', fromlist=['families']).families.Poisson()).fit(cov_type='cluster', cov_kwds={'groups': groups})
lp = smf.ols('hit ~ C(year) + C(fbin) + C(sido) + C(month) + run10', data=S).fit(cov_type='cluster', cov_kwds={'groups': groups})
R['irr'] = {}; R['p_any_diff'] = {}
for y in [2025, 2026]:
    k = f'C(year)[T.{y}]'; b, se = m.params[k], m.bse[k]
    R['irr'][y] = [round(float(np.exp(b)), 3), round(float(np.exp(b - 1.96 * se)), 3), round(float(np.exp(b + 1.96 * se)), 3)]
    R['p_any_diff'][y] = [round(float(lp.params[k]), 3), round(float(1.96 * lp.bse[k]), 3)]
nb = smf.negativebinomial('heat ~ C(year) + C(fbin) + C(sido) + C(month) + run10', data=S).fit(disp=0, maxiter=300, cov_type='cluster', cov_kwds={'groups': groups})
R['irr_nb'] = {}
for y in [2025, 2026]:
    k = f'C(year)[T.{y}]'; b, se = nb.params[k], nb.bse[k]
    R['irr_nb'][y] = [round(float(np.exp(b)), 3), round(float(np.exp(b - 1.96 * se)), 3), round(float(np.exp(b + 1.96 * se)), 3), round(float(nb.pvalues[k]), 3)]
# 대안: 시도 안 관측소 '평균' 체감온도로 바꿔도 같은가
S['fbin_mean'] = np.clip(np.floor(S.feels_mean), 26, 37).astype(int)
mm = smf.glm('heat ~ C(year) + C(fbin_mean) + C(sido) + C(month) + run10', data=S.dropna(subset=['feels_mean']),
             family=__import__('statsmodels.api', fromlist=['families']).families.Poisson()).fit(cov_type='cluster', cov_kwds={'groups': S.dropna(subset=['feels_mean']).sido.astype('category').cat.codes})
b, se = mm.params['C(year)[T.2025]'], mm.bse['C(year)[T.2025]']
R['irr_meanfeels_2025'] = [round(float(np.exp(b)), 3), round(float(np.exp(b - 1.96 * se)), 3), round(float(np.exp(b + 1.96 * se)), 3)]
R['p_any_pvalue'] = {y: round(float(lp.pvalues[f'C(year)[T.{y}]']), 3) for y in [2025, 2026]}
hs = S[S.hot == 1].groupby(['sido', 'year']).heat.mean().unstack()
R['ratio_2025_2024_by_sido'] = (hs[2025] / hs[2024]).round(2).sort_values(ascending=False).to_dict()
print('문자 수 배율(2024 대비) 포아송:', R['irr'], ' 음이항:', R['irr_nb']); print('보낼지 여부 확률 차이:', R['p_any_diff'], R['p_any_pvalue'])
print('시도별 2025/2024 배율:', R['ratio_2025_2024_by_sido'])

## Step 8. 늘어난 문자는 어디서 왔나: 같은 날 두 번 이상 보낸 시군구

In [ ]:
h = heat.copy(); h['year'] = h.date.str[:4].astype(int)
h['sgg'] = h.RCPTN_RGN_NM.str.strip().str.split().str[:2].str.join(' ')
per = h[h.year.isin([2024, 2025, 2026])].groupby(['year', 'date', 'sgg']).size()
R['repeat_share'] = {int(k): round(float(v), 3) for k, v in (per >= 2).groupby('year').mean().items()}
R['sgg_sending'] = {int(k): int(v) for k, v in h[h.year.isin([2024, 2025, 2026])].groupby('year').sgg.nunique().items()}
R['msgs_per_sgg_day'] = {int(k): round(float(v), 2) for k, v in per.groupby('year').mean().items()}
hd = h[h.year.isin([2024, 2025, 2026])].drop_duplicates(['date', 'sgg', 'msg_hash'])
R['repeat_share_dedup'] = {int(k): round(float(v), 3) for k, v in (hd.groupby(['year', 'date', 'sgg']).size() >= 2).groupby('year').mean().items()}
print(R['repeat_share'], R['repeat_share_dedup'], R['sgg_sending'])

## Step 9. 지역별 차이와 비(호우), 시간대

In [ ]:
reg = hot.groupby('sido').agg(per_hot_day=('heat', 'mean'), hot_days=('heat', 'size'), p_any=('hit', 'mean')).sort_values('per_hot_day')
# 시군구(기초자치단체) 수: 2023년 7월 군위군 대구 편입 이후 기준, 합계 226. 세종·제주는 기초자치단체가 없어 1·2(행정시)로 둡니다.
N_SGG = {'서울': 25, '부산': 16, '대구': 9, '인천': 10, '광주전남': 27, '대전': 5, '울산': 5, '세종': 1, '경기': 31,
         '강원': 18, '충북': 11, '충남': 15, '전북': 14, '경북': 22, '경남': 18, '제주': 2}
reg['n_sgg'] = pd.Series(N_SGG); reg['per_sgg'] = reg.per_hot_day / reg.n_sgg
reg = reg.sort_values('per_sgg')
R['region'] = json.loads(reg.round(2).to_json(orient='index'))
h3 = h[h.year.isin([2024, 2025, 2026])]
R['top_heat_senders'] = h3.RCPTN_RGN_NM.str.strip().value_counts().head(5).to_dict()
dg = h3[h3.sido_list == '대구']
R['daegu_heat'] = {'total': int(len(dg)), **{k: int(v) for k, v in dg.RCPTN_RGN_NM.str.strip().value_counts().head(3).items()}}
R['heat_step_share'] = h3.EMRG_STEP_NM.value_counts(normalize=True).round(4).to_dict()
print(R['top_heat_senders'], R['daegu_heat'])
Rn = panel.copy(); Rn['hit'] = (Rn.rain_alert > 0).astype(int)
Rn['rbin'] = pd.cut(Rn.rain, [-1, 0.1, 10, 30, 60, 100, 1000], right=False,
                    labels=['0', '0.1~10', '10~30', '30~60', '60~100', '100 이상'])
rain = Rn.groupby('rbin', observed=True).hit.agg(['mean', 'size'])
R['rain'] = {str(k): [round(float(v['mean']), 3), int(v['size'])] for k, v in rain.iterrows()}
hour = alerts.assign(kind=np.where(alerts.DST_SE_NM == '폭염', '폭염', '그 밖의 유형')).groupby(['hour', 'kind']).size().unstack(fill_value=0)
R['hour_heat_share_10'] = round(float(hour.loc[10, '폭염'] / hour.loc[10].sum()), 3)
R['hour_peak'] = int(hour.sum(axis=1).idxmax())
print(reg.round(2)); print(rain)
json.dump(R, open(os.path.join(HERE, 'results.json'), 'w'), ensure_ascii=False, indent=1, default=str)

## Step 10. 차트 그리기 (블로그 이미지와 같은 스타일)

In [ ]:
BG, INK, MUTE, GRID, RED, GREY = '#F5F5F0', '#1F2328', '#6B6F76', '#DADAD2', '#C8405A', '#8A9096'
def frame(kicker, title, sub, source):
    fig = plt.figure(figsize=(12, 7.2), dpi=200, facecolor=BG)
    fig.text(0.055, 0.935, kicker, color=RED, fontsize=13, fontweight='bold')
    fig.text(0.055, 0.855, title, color=INK, fontsize=25, fontweight='bold')
    fig.text(0.055, 0.795, sub, color=MUTE, fontsize=13)
    fig.add_artist(plt.Line2D([0.055, 0.945], [0.085, 0.085], color=GRID, lw=1))
    fig.text(0.055, 0.04, source, color=MUTE, fontsize=11)
    ax = fig.add_axes([0.1, 0.17, 0.84, 0.55], facecolor=BG)
    for s in ['top', 'right', 'left']: ax.spines[s].set_visible(False)
    ax.spines['bottom'].set_color(GRID); ax.tick_params(colors=MUTE, labelsize=12, length=0)
    ax.grid(axis='y', color=GRID, lw=0.8); ax.set_axisbelow(True)
    return fig, ax
SRC = '출처: 행정안전부 긴급재난문자(재난안전데이터공유플랫폼), 기상청 ASOS 일자료(공공데이터포털) · 데이터팝콘 재계산'

# 01 데이터 공백
d = daily['2025-08-15':'2025-11-30']
fig, ax = frame('01 / 데이터 공백', '2025년 9월 26일부터 31일 동안 기록이 0건입니다',
                '하루 단위 재난문자 기록 수. 같은 날 밤 국가정보자원관리원 대전센터에서 화재가 났습니다.', SRC)
x = pd.to_datetime(d.index)
ax.bar(x, d.values, color=[RED if GAP[0] <= k <= GAP[1] else GREY for k in d.index], width=0.8)
ax.axvspan(pd.Timestamp(GAP[0]) - pd.Timedelta(hours=12), pd.Timestamp(GAP[1]) + pd.Timedelta(hours=12), color=RED, alpha=0.08)
ax.text(pd.Timestamp('2025-10-11'), d.max() * 0.62, '0건\n(9/26~10/26)', color=RED, fontsize=15, fontweight='bold', ha='center')
ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter('%m/%d'))
fig.savefig(f'{OUT}/01-data-gap.png', facecolor=BG); plt.show()

# 02 체감온도별 확률
fig, ax = frame('02 / 날씨와 문자', '체감 33도를 넘는 날, 10곳 중 9곳에 폭염 문자가 왔습니다',
                '2024~2026년 6~9월, 시도×일 기준. 체감온도는 기상청 여름철 공식으로 일최고기온·최소습도에서 근사했습니다.', SRC)
lab = [f'{b}' if 26 < b < 37 else ('26 이하' if b == 26 else '37 이상') for b in dose.fbin]
ax.bar(range(len(dose)), dose.p * 100, color=[RED if b >= 33 else GREY for b in dose.fbin], width=0.7)
for i, (p, n) in enumerate(zip(dose.p, dose.n)):
    ax.text(i, p * 100 + 2, f'{p*100:.0f}%', ha='center', color=INK, fontsize=11)
ax.set_xticks(range(len(dose))); ax.set_xticklabels(lab); ax.set_ylim(0, 110); ax.set_xlabel('그날 시도 최고 체감온도(℃)', color=MUTE, fontsize=12)
ax.set_yticks([0, 25, 50, 75, 100]); ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
fig.savefig(f'{OUT}/02-heat-dose.png', facecolor=BG); plt.show()

# 03 같은 더위, 더 많은 문자
fig, ax = frame('03 / 같은 더위, 다른 문자 수', '같은 체감온도에서도 2025년부터 문자가 더 많이 왔습니다',
                '시도 하루에 받은 폭염 문자 수(평균). 선 하나가 한 해 여름(6~9월)입니다.', SRC)
cols = {2024: GREY, 2025: RED, 2026: '#E08A9B'}
xs = [b for b in curve.index if b >= 29]
for y in [2024, 2025, 2026]:
    ys = curve.loc[xs, y].values
    ax.plot(xs, ys, color=cols[y], lw=3, marker='o', ms=6)
    ax.text(xs[-1] + 0.15, ys[-1], f'{y}', color=cols[y], fontsize=13, fontweight='bold', va='center')
ax.set_xticks(xs); ax.set_xticklabels([f'{b}' if b < 36 else '36 이상' for b in xs]); ax.set_xlabel('그날 시도 최고 체감온도(℃)', color=MUTE, fontsize=12)
irr = R['irr'][2025]
nbi = R['irr_nb'][2025]
ax.text(xs[0], curve.loc[xs].max().max() * 0.86, f'기온·지역·달·폭염 지속일을 같게 놓으면 2025년은 2024년의\n포아송 모형 {irr[0]:.2f}배 (95% 구간 {irr[1]:.2f}~{irr[2]:.2f})\n음이항 모형 {nbi[0]:.2f}배 (95% 구간 {nbi[1]:.2f}~{nbi[2]:.2f}, 유의하지 않음)', color=INK, fontsize=12)
fig.savefig(f'{OUT}/03-same-heat-more-texts.png', facecolor=BG); plt.show()

# 04 같은 날 두 번 이상
fig, ax = frame('04 / 늘어난 이유', '같은 날 폭염 문자를 두 번 이상 보낸 시군구가 두 배가 됐습니다',
                '폭염 문자를 보낸 시군구-일 가운데 같은 날 2건 이상 보낸 비율. 보낸 시군구 수는 해마다 비슷합니다.', SRC)
ys = [R['repeat_share'][y] * 100 for y in [2024, 2025, 2026]]
ax.bar([0, 1, 2], ys, color=[GREY, RED, RED], width=0.55)
for i, (v, y) in enumerate(zip(ys, [2024, 2025, 2026])):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', color=INK, fontsize=15, fontweight='bold')
    ax.text(i, -4.5, f'보낸 시군구 {R["sgg_sending"][y]}곳', ha='center', color=MUTE, fontsize=11)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(['2024', '2025', '2026']); ax.set_ylim(0, 40); ax.set_yticks([0, 10, 20, 30, 40]); ax.set_yticklabels(['0%', '10%', '20%', '30%', '40%'])
fig.savefig(f'{OUT}/04-repeat-sends.png', facecolor=BG); plt.show()

# 05 지역 (시군구 하나당)
fig, ax = frame('05 / 지역 차이', '같은 더위에도 시군구마다 보내는 습관이 다릅니다',
                '체감 33도 이상인 날, 시군구 하나가 하루에 보낸 폭염 문자 수(2024~2026년 6~9월). 시도 합계를 시군구 수로 나눴습니다.', SRC)
ax.barh(range(len(reg)), reg.per_sgg, color=[RED if s in ('경북', '대구') else GREY for s in reg.index], height=0.65)
for i, (v, n) in enumerate(zip(reg.per_sgg, reg.n_sgg)): ax.text(v + 0.01, i, f'{v:.2f}  (시군구 {int(n)})', va='center', color=INK, fontsize=10.5)
ax.set_yticks(range(len(reg))); ax.set_yticklabels(reg.index); ax.grid(axis='y', visible=False); ax.grid(axis='x', color=GRID)
ax.set_xlim(0, reg.per_sgg.max() * 1.3)
fig.savefig(f'{OUT}/05-region.png', facecolor=BG); plt.show()

# 06 비
fig, ax = frame('06 / 비와 문자', '비는 문자와 거의 정확히 맞습니다',
                '그날 시도 관측소 최대 일강수량 구간별로 호우 문자가 1건 이상 온 비율(2023.9~2026.10, 공백 기간 제외).', SRC)
keys = list(R['rain'].keys()); vals = [R['rain'][k][0] * 100 for k in keys]
ax.bar(range(len(keys)), vals, color=[RED if k in ('60~100', '100 이상') else GREY for k in keys], width=0.6)
for i, v in enumerate(vals): ax.text(i, v + 2, f'{v:.0f}%', ha='center', color=INK, fontsize=13, fontweight='bold')
ax.set_xticks(range(len(keys))); ax.set_xticklabels(['0mm', '0.1~10mm', '10~30mm', '30~60mm', '60~100mm', '100mm 이상']); ax.set_ylim(0, 112); ax.set_yticks([0, 25, 50, 75, 100]); ax.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
fig.savefig(f'{OUT}/06-rain.png', facecolor=BG); plt.show()

# 07 시간대
fig, ax = frame('07 / 시간대', '오전 10시 정점은 폭염 문자가 만듭니다',
                '발송 시각별 재난문자 수(전체 63,402건). 빨간색이 폭염 문자입니다.', SRC)
ax.bar(hour.index, hour['그 밖의 유형'], color=GREY, width=0.75)
ax.bar(hour.index, hour['폭염'], bottom=hour['그 밖의 유형'], color=RED, width=0.75)
ax.set_xticks(range(0, 24, 3)); ax.set_xticklabels([f'{h}시' for h in range(0, 24, 3)])
fig.savefig(f'{OUT}/07-hour.png', facecolor=BG); plt.show()
print('차트 저장:', sorted(os.listdir(OUT)))